# Ask VCARE: live demo

**What this does, in plain terms:** you type a question, and an AI assistant answers it by calling the **real deployed VCARE CC Compass app** (`vcare-pathway-checklist`) over the internet, signed in as the hackathon test account **Inspector Gadget**. It is the same connection that scored 10/10 in the live evaluation.

**What you see for each question (the answer card):**
* **Question**: what was asked
* **Tool used**: which of the 3 tools the AI picked, and what it sent
* **Answer**: in plain English
* **Why it matters / next step**: a short clinical-review next step and where to look on the published dashboard **VCARE CC Compass: Facility High-Risk Findings**
* **Details from the tool**: the data table (Genie), the flags (screening), or the checklist questions and sources
* **Time**: how many seconds it took

**How to run it (about 1 minute to set up):**
1. Click **Run all**. This installs libraries, signs in and checks that all 3 tools are reachable.
2. Type a question in the **question** box at the top.
3. Re-run the **Ask** cell (Step 5). Or run one of the ready examples (Steps 6 to 8).

**Tip for the presentation:** Genie questions take 20 to 40 seconds because Genie writes and runs a database query. Keep talking while it works.

*All data is synthetic, including consult numbers. Results are review priorities, not clinical determinations.*

In [0]:
# Installs the MCP client and tracing libraries, then restarts Python so they load cleanly.
# (The yellow "you may need to restart" note is normal: the next line already restarts.)
%pip install -q --upgrade "databricks-mcp" "databricks-sdk>=0.60" "mlflow>=3.4" "openai>=1.40" nest_asyncio
%restart_python

In [0]:
# Settings for the demo. Change these values, not the code below.
APP_URL = "https://vcare-pathway-checklist-7474654188777746.aws.databricksapps.com"  # the app's URL, WITHOUT /mcp
AGENT_MODEL = "databricks-claude-sonnet-4-5"                     # same assistant as the live evaluation
EXPERIMENT = "/Users/anna.fernandez@va.gov/mcp_vcare_compass_demo"  # demo traces go here (kept apart from the scored eval runs)
DASHBOARD_NAME = "VCARE CC Compass: Facility High-Risk Findings"
DASHBOARD_URL = "https://dbc-aae0ba16-8727.cloud.databricks.com/dashboardsv3/01f1d75336601c39a305ecd4c8a9ada5/published?o=7474654188777746"
EXPECTED_TOOL_NAMES = ["ask_vcare_case_data", "flag_high_risk_case", "get_consult_pathway_checklist"]
DEFAULT_QUESTION = "Which facility has the most missed 72-hour ER notifications?"

# The question box at the top of the notebook. Type here, then re-run Step 5.
dbutils.widgets.text("question", DEFAULT_QUESTION, "Type your question")

# Standing instructions for the assistant during the demo.
SYSTEM_PROMPT = f"""You help VA Community Care consult reviewers. Answer using ONLY the provided tools; never invent numbers.
All data is synthetic. Your output supports human review priorities; it is never a clinical determination.

Tool guide:
- ask_vcare_case_data: questions about counts, facilities, trends or lists in the synthetic consult data.
- flag_high_risk_case: screen ONE described case against the configured review rules.
- get_consult_pathway_checklist: review questions and policy sources for a consult pathway step
  (receive, triage, eligibility, authorize, schedule, follow_up, close).

After using the tools, reply in EXACTLY this format, plain English, short enough to read aloud:
ANSWER: <1-3 sentences with the key fact or number>
WHY IT MATTERS: <1-2 sentences on the risk to Veterans or timeliness>
NEXT STEP: <one concrete clinical-review next step (who should do what), then: "See the {DASHBOARD_NAME} dashboard" and name the page or widget to look at (for example the facility table, the Priority 1 ER list, the idle high-risk list, or the Staffing & Targets page)>
"""
print("Settings ready. Question box default:", DEFAULT_QUESTION)

In [0]:
# Signs in as the hackathon test account and lists the tools the LIVE app offers.
# The sign-in secret stays in Unity Catalog (models.ai / inspector-gadget); it is never printed.
import json, time, asyncio
from datetime import timedelta
from urllib.parse import urljoin
import httpx
import nest_asyncio
import mcp.client.streamable_http as _streamable_http
from mcp import ClientSession
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
nest_asyncio.apply()

def inspector_gadget_client() -> WorkspaceClient:
    params = json.loads(dbutils.secrets.get(catalog="models", schema="ai", key="inspector-gadget"))
    host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    return WorkspaceClient(host=host, client_id=params["client_id"], client_secret=params["client_secret"])

ig_client = inspector_gadget_client()
SERVER_URL = urljoin(APP_URL, "mcp")

# Listing tools is quick, so the standard client is fine here.
live_tools = DatabricksMCPClient(server_url=SERVER_URL, workspace_client=ig_client).list_tools()
names = sorted(t.name for t in live_tools)
print("Tools on the live app:", names)
assert names == sorted(EXPECTED_TOOL_NAMES), f"Expected {EXPECTED_TOOL_NAMES}, got {names}. Is the app running?"
print("PASS  all 3 tools are reachable on the live app")

# Tool calls use the standard MCP client with longer timeouts (120 s to connect/send, 300 s to read):
# Genie needs 10-40 s to write and run its query, longer than DatabricksMCPClient's ~5 s default.
def _auth_headers():
    """A fresh Inspector Gadget sign-in header (OAuth token) for each call; never printed."""
    return ig_client.config.authenticate()

async def _session_call(streams, name, args):
    async with ClientSession(streams[0], streams[1]) as session:
        await session.initialize()
        return await session.call_tool(name, args)

async def _call_tool_async(name, args):
    if hasattr(_streamable_http, "streamablehttp_client"):   # older mcp library
        async with _streamable_http.streamablehttp_client(SERVER_URL, headers=_auth_headers(), timeout=timedelta(seconds=120),
                                                          sse_read_timeout=timedelta(seconds=300)) as streams:
            return await _session_call(streams, name, args)
    # newer mcp library: the timeouts live on the web client
    async with httpx.AsyncClient(headers=_auth_headers(), timeout=httpx.Timeout(120.0, read=300.0), follow_redirects=True) as http:
        async with _streamable_http.streamable_http_client(SERVER_URL, http_client=http) as streams:
            return await _session_call(streams, name, args)

def call_tool_live(name, args):
    """Call one tool on the live app as Inspector Gadget, allowing up to 5 minutes for a slow reply."""
    return asyncio.run(_call_tool_async(name, args))

# Warm-up: one fast real call so the app is awake before the audience is watching.
t0 = time.time()
warm = call_tool_live("get_consult_pathway_checklist", {"args": {"pathway_step": "authorize"}})
print(f"PASS  warm-up call answered in {time.time() - t0:.1f} s" if warm.content else "WARN  warm-up call returned nothing")

In [0]:
# The assistant loop (same as the live evaluation) plus a clean answer card for the audience.
# MLflow tracing is on, so every question can be replayed step by step in the Traces tab.
import html, re
import pandas as pd
import mlflow
from mlflow.entities import SpanType

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(EXPERIMENT)
mlflow.openai.autolog()

def _input_schema(tool):
    """The tool's input format. Different MCP client versions name this attribute differently."""
    for attr in ("inputSchema", "input_schema", "parameters"):
        schema = getattr(tool, attr, None)
        if schema:
            return schema if isinstance(schema, dict) else json.loads(json.dumps(schema, default=lambda o: o.__dict__))
    dumped = tool.model_dump() if hasattr(tool, "model_dump") else vars(tool)
    for key in ("inputSchema", "input_schema", "parameters"):
        if dumped.get(key):
            return dumped[key]
    raise AttributeError(f"Could not find the input format for tool {tool.name}; available fields: {list(dumped)}")

TOOLS = [{"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": _input_schema(t)}}
         for t in live_tools]
llm = WorkspaceClient().serving_endpoints.get_open_ai_client()   # the AI model uses YOUR sign-in; only the app uses Inspector Gadget

def live_tool_text(name, args):
    """Run one tool on the live app and return its text. Errors go back to the assistant, not raised."""
    try:
        result = call_tool_live(name, args)
        return result.content[0].text if result.content else ""
    except Exception as e:  # network errors arrive wrapped in an ExceptionGroup; report the real cause
        inner = getattr(e, "exceptions", [e])
        return json.dumps({"error": " | ".join(f"{type(x).__name__}: {x}" for x in inner)[:500]})

@mlflow.trace(span_type=SpanType.AGENT, name="ask_vcare_live")
def run_assistant(question: str) -> dict:
    """Let the assistant work the question with the live tools. Returns the reply and every tool call."""
    mlflow.update_current_trace(tags={"purpose": "live demo", "transport": "live app as Inspector Gadget"})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    calls = []
    for _ in range(6):   # at most 6 rounds of tool use
        msg = llm.chat.completions.create(model=AGENT_MODEL, messages=messages, tools=TOOLS, temperature=0).choices[0].message
        if not msg.tool_calls:
            return {"reply": (msg.content or "").strip(), "calls": calls}
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            started = time.time()
            with mlflow.start_span(name=tc.function.name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                output = live_tool_text(tc.function.name, args)
                span.set_outputs(output[:4000])
            calls.append({"tool": tc.function.name, "args": args, "seconds": round(time.time() - started, 1), "output": output})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output[:12000]})
    return {"reply": "ANSWER: (no final answer within 6 rounds; please rephrase the question)", "calls": calls}

# ---------- answer card ----------
def _md(text):
    """Safe HTML from the assistant's text, keeping **bold** and line breaks."""
    t = html.escape(text or "").strip()
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t).replace("\n", "<br>")

def _sections(reply):
    """Split the reply into ANSWER / WHY IT MATTERS / NEXT STEP (falls back to the whole text)."""
    parts = {}
    for key in ("ANSWER", "WHY IT MATTERS", "NEXT STEP"):
        m = re.search(rf"{key}\s*:\s*(.*?)(?=\n\s*(?:ANSWER|WHY IT MATTERS|NEXT STEP)\s*:|\Z)", reply, re.S | re.I)
        parts[key] = m.group(1).strip() if m else ""
    if not parts["ANSWER"]:
        parts["ANSWER"] = reply
    return parts

def _parse(output):
    try:
        return json.loads(output)
    except (TypeError, ValueError):
        return {}

def _tool_details(call):
    """Extra value from the tool itself, beyond the one-line answer."""
    data, name = _parse(call["output"]), call["tool"]
    if data.get("error"):
        return f"<div style='color:#b00020'>Tool error: {html.escape(str(data['error']))[:400]}</div>"
    if name == "ask_vcare_case_data":
        bits = []
        if data.get("rows") and data.get("columns"):
            df = pd.DataFrame(data["rows"][:10], columns=data["columns"])
            total = data.get("total_row_count") or len(data["rows"])
            bits.append(f"<div><b>Data from Genie</b> (showing {len(df)} of {total} rows)</div>" + df.to_html(index=False, border=0, classes='vc'))
        if data.get("suggested_questions"):
            bits.append("<div><b>Good follow-up questions:</b><ul>" + "".join(f"<li>{html.escape(q)}</li>" for q in data["suggested_questions"][:3]) + "</ul></div>")
        return "".join(bits)
    if name == "flag_high_risk_case":
        if not data.get("flags"):
            return "<div><b>Screening result:</b> no flags for this case under the configured rules.</div>"
        rows = "".join(f"<tr><td><b>{html.escape(f['severity'])}</b></td><td>{html.escape(f['code'])}</td><td>{html.escape(f['finding'])}</td><td>{html.escape(f['next_step'])}</td></tr>" for f in data["flags"])
        th = html.escape(json.dumps(data.get("thresholds_used", {})))
        return ("<div><b>Flags raised</b></div><table class='vc'><tr><th>Severity</th><th>Code</th><th>Finding</th><th>Next step</th></tr>" + rows + "</table>"
                f"<div style='font-size:12px;color:#555'>Thresholds used: {th}</div>")
    if name == "get_consult_pathway_checklist":
        out = []
        for s in data.get("steps", [])[:2]:
            out.append(f"<div><b>{html.escape(s['label'])}:</b> {html.escape(s['description'])}</div><ul>" + "".join(f"<li>{html.escape(q)}</li>" for q in s["gemba_questions"]) + "</ul>"
                       f"<div style='font-size:12px;color:#555'>Sources: {html.escape('; '.join(s['sources']))}</div>")
        return "".join(out)
    return ""

def ask(question: str):
    """Ask one question through the live app and show the answer card."""
    question = (question or "").strip()
    if not question:
        print("Type a question in the 'question' box at the top first."); return
    started = time.time()
    result = run_assistant(question)
    seconds = round(time.time() - started, 1)
    s = _sections(result["reply"])
    try:
        trace_id = mlflow.get_last_active_trace_id()
    except Exception:
        trace_id = None
    if result["calls"]:
        tools_html = "".join(
            f"<div><code>{html.escape(c['tool'])}</code> ({c['seconds']} s) sent: <code>{html.escape(json.dumps(c['args'].get('args', c['args'])))[:400]}</code></div>"
            for c in result["calls"])
    else:
        tools_html = "<div>(no tool used)</div>"
    details = "".join(_tool_details(c) for c in result["calls"])
    card = f"""
<style>.vc{{border-collapse:collapse;font-size:13px;margin:6px 0}} .vc td,.vc th{{border:1px solid #9ec5e8;padding:4px 8px;text-align:left}} .vc th{{background:#e8f1fa}}</style>
<div style="font-family:Arial;border:2px solid #003F72;border-radius:8px;max-width:1000px">
  <div style="background:#003F72;color:white;padding:10px 14px;font-size:18px"><b>Ask VCARE</b> &nbsp;|&nbsp; live app</div>
  <div style="padding:12px 14px;line-height:1.45">
    <div style="font-size:16px"><b>Question:</b> {html.escape(question)}</div>
    <div style="margin-top:8px;color:#333"><b>Tool used:</b>{tools_html}</div>
    <div style="margin-top:10px;background:#e8f1fa;border-left:5px solid #0071BC;padding:10px;font-size:16px"><b>Answer:</b> {_md(s['ANSWER'])}</div>
    <div style="margin-top:10px"><b>Why it matters:</b> {_md(s['WHY IT MATTERS']) or '-'}</div>
    <div style="margin-top:6px"><b>Next step:</b> {_md(s['NEXT STEP']) or '-'} &nbsp;<a href="{DASHBOARD_URL}" target="_blank">Open the dashboard</a></div>
    <div style="margin-top:10px">{details}</div>
    <div style="margin-top:10px;font-size:12px;color:#555">Answered in {seconds} s{' &nbsp;|&nbsp; MLflow trace ' + html.escape(trace_id) if trace_id else ''}</div>
    <div style="margin-top:4px;font-size:12px;color:#555"><i>All data is synthetic, including consult numbers. Results are review priorities, not clinical determinations.</i></div>
  </div>
</div>"""
    displayHTML(card)

print("Assistant ready with tools:", [t["function"]["name"] for t in TOOLS])
print("Demo traces go to MLflow experiment:", EXPERIMENT)

In [0]:
# Reads the question box at the top of the notebook and answers it through the live app.
ask(dbutils.widgets.get("question"))

In [0]:
# Data question answered by Genie through the live app. Takes about 20 to 40 seconds.
# Expected (synthetic data): Hawkeye Ridge, with all 10 missed 72-hour ER notifications.
ask("Which facility has the most missed 72-hour ER notifications?")

In [0]:
# Screens one made-up case against the configured review rules. Takes about 5 seconds.
# The screening tool works from dates and coordination level (it has no triage-delay or care-options fields),
# so the "triage took 9 days" idea is described as 9 days with no nurse follow-up.
# Expected: a High flag (COORD_TIMELINESS) because 9 days is over the 3-day Urgent follow-up threshold.
ask("Screen synthetic case SYN-DEMO-01 as of 2026-11-03: Radiology, Urgent coordination level, active consult, "
    "high-risk marker present, last nurse action on 2026-10-25 (9 days ago). Care options were not documented. "
    "What flags does it raise and what should the reviewer do?")

In [0]:
# Review questions and policy sources for the Authorize step of the consult pathway. Takes about 5 seconds.
ask("What should a reviewer check at the authorization step of a community care consult?")

**To show the AI's work step by step:** open **Experiments** in the left sidebar, then **mcp_vcare_compass_demo**, then the **Traces** tab. Click the newest trace to see the question, the tool call sent to the live app, the tool's reply, and the final answer.

**If something goes wrong**
* **Genie answer is slow:** normal (20 to 40 seconds). Keep talking.
* **HTTP 401 or 403:** Inspector Gadget's access to the app changed. Check Compute, then Apps, then `vcare-pathway-checklist`, then Permissions.
* **"Type a question first":** the question box is empty.
* **Anything else red:** run Step 3 again, then re-run Step 5. Use your backup screenshots if needed.

*All data is synthetic, including consult numbers. Results are review priorities, not clinical determinations.*